# Flux-Corrected Diagonal Frog — Reproduction Notebook
**ArXivist-generated reproduction notebook**

Paper: Andrey Itkin, *"Flux-Corrected Diagonal Frog: second order and positivity at all time steps"*, [arXiv:2607.20415v1](https://arxiv.org/abs/2607.20415) (math.NA), July 2026.

Generated: 2026-07-26

This is a **numerical-PDE scheme paper**, not a machine-learning paper — there is no
learned model, no dataset, no training loop. Every "result" below is a deterministic
banded linear-algebra computation. This notebook walks through the core operator
construction, the flux limiter, all six solver variants, runs a small end-to-end
reproduction, and compares against the paper's reported Tables.


In [ ]:
# Environment check (CPU-only; no GPU needed anywhere in this paper)
import sys, platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
try:
    import numpy, scipy, matplotlib, pandas
    print(f"numpy: {numpy.__version__}")
    print(f"scipy: {scipy.__version__}")
    print(f"matplotlib: {matplotlib.__version__}")
    print(f"pandas: {pandas.__version__}")
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Run: pip install -r ../requirements.txt")


In [ ]:
# Install the project in editable mode (run once)
import subprocess, sys, os
try:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".."],
                             capture_output=True, text=True, cwd=os.getcwd())
    print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])
except Exception as e:
    print(f"Install step failed ({e}); if fcdf_diagonal_frog is already importable, this is safe to ignore.")


## Paper Overview

**Problem**: linear second-order finite-difference schemes for the Fokker-Planck equation
cannot preserve positivity (Godunov's theorem). The prior "Diagonal Frog" (DF) framework
used *eventual positivity* (matrix-exponential / resolvent maps become entrywise
nonnegative only above an explicit step-size threshold), but that leaves small steps
uncovered.

**Core idea**: split the full second-order operator $A_2$ into a monotone M-matrix core
$A_1$ (centered diffusion + 1st-order upwind convection) and an antidiffusive correction
$C$:
$$A_2 = A_1 + C$$
Then apply a **Zalesak-type flux limiter** per interface, clamping the antidiffusive
correction just enough to guarantee the right-hand side of every implicit sweep stays
nonnegative — **for every step size**, not just above some threshold. This is Scheme
**FCDF-B**, the paper's primary contribution (Proposition 1): unconditionally positive,
exactly mass-conservative, and 2nd-order accurate wherever the limiter is inactive.

**Extensions**: a defect-corrected 2-stage scheme (**FCDF-DC**) restores 2nd order *in
time* as well; an **active-set semismooth-Newton solver** removes the Picard iteration's
step-size restriction entirely, at a cost governed by the number of nodes where positivity
binds rather than by the step size.

**Implementation map**:
- Section 2 (operator split) → `src/fcdf_diagonal_frog/operators/df_operator.py`
- Section 3 (FCDF-A, FCDF-B, limiter) → `schemes/fcdf_a.py`, `fcdf_b.py`, `limiter/zalesak.py`
- Section 4 (FCDF-DC) → `schemes/fcdf_dc.py`
- Section 5 (active-set solver, coverage) → `schemes/active_set.py`, `linear_windows/thresholds.py`
- Section 6 (benchmarks, all tables) → `benchmarks/`, `evaluate.py`


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
import numpy as np
import matplotlib.pyplot as plt

from fcdf_diagonal_frog.operators.grid import Grid1D
from fcdf_diagonal_frog.operators.df_operator import DFOperator

print("Core imports OK")


## Component 1: The Grid and Core-Correction Split (Section 2)

$$A_2 = A_1 + C, \qquad A_1 \text{ monotone M-matrix (Eq. 7)}, \qquad C = A_2 - A_1 \text{ antidiffusive, no diffusion (Eq. 8-9)}$$

Both are assembled via conservative interface-flux differencing with a zero-flux boundary
closure, guaranteeing $\mathbf{1}^\top A_1 = \mathbf{1}^\top A_2 = \mathbf{1}^\top C = 0$
(exact mass conservation) by construction.


In [ ]:
# Toy assembly: uniform positive drift + diffusion on [0,1], n=20
grid = Grid1D(x_min=0.0, x_max=1.0, n=20)
mu = np.full(grid.n, 1.0)
D = np.full(grid.n, 0.05)

op = DFOperator.assemble(grid, mu, D)

print(f"Grid: {grid}")
print(f"A1 shape: {op.A1.shape}, nnz: {op.A1.nnz}")
print(f"A2 shape: {op.A2.shape}, nnz: {op.A2.nnz}")
print(f"1^T A1 max abs col sum (should be ~0): {np.max(np.abs(np.asarray(op.A1.sum(axis=0)))):.2e}")
print(f"1^T A2 max abs col sum (should be ~0): {np.max(np.abs(np.asarray(op.A2.sum(axis=0)))):.2e}")
print(f"1^T C  max abs col sum (should be ~0): {np.max(np.abs(np.asarray(op.C.sum(axis=0)))):.2e}")


## Component 2: The Zalesak Limiter (Section 3, Eq. 15)

$$\theta_{i+1/2} = \min\left(1, \frac{h\,b_{j(i+1/2)}}{2\,\gamma\,|d_{i+1/2}(p)|}\right)$$

Equivalently, a clamp of the unlimited antidiffusive flux onto $[-c^-_{i+1/2}, c^+_{i+1/2}]$
with $c^\pm = h\,b/(2\gamma)$ (Lemma 4). This is what makes Scheme FCDF-B unconditionally
positive.


In [ ]:
from fcdf_diagonal_frog.limiter.zalesak import ZalesakLimiter

limiter = ZalesakLimiter(kappa=2)
b = np.where((grid.x >= 0.3) & (grid.x <= 0.6), 1.0, 0.0)  # a plateau
gamma = 0.05
d_unlimited = op.unlimited_flux(b)
Lambda = limiter.clamp(d_unlimited, b, gamma, grid.h)

print(f"Unlimited flux range: [{d_unlimited.min():.3f}, {d_unlimited.max():.3f}]")
print(f"Limited flux range:   [{Lambda.min():.3f}, {Lambda.max():.3f}]  (clamped)")


## Component 3: Scheme FCDF-B — the paper's primary scheme (Proposition 1)

Run a single implicit step on a plateau (unresolved-front-like) initial condition and
verify: (i) positivity, (ii) exact mass conservation.


In [ ]:
from fcdf_diagonal_frog.schemes.fcdf_b import FCDF_B_Solver
from fcdf_diagonal_frog.schemes.unlimited import UnlimitedSolver

solver = FCDF_B_Solver()
unl = UnlimitedSolver()

out = solver.step(op, limiter, b, gamma)
p_fcdfb = out["p"]
p_unlimited = unl.step(op, b, gamma)

print(f"FCDF-B:    min(p)={p_fcdfb.min():.3e}   mass={np.sum(p_fcdfb)*grid.h:.6f}  (sweeps={out['sweeps']})")
print(f"Unlimited: min(p)={p_unlimited.min():.3e}   mass={np.sum(p_unlimited)*grid.h:.6f}")
print(f"\nOriginal mass: {np.sum(b)*grid.h:.6f}")


## Component 4: FCDF-DC — 2nd order in time (Proposition 3)

Predictor-corrector using only limited backward-Euler-type solves, never an explicit
(sign-indefinite) half-step.


In [ ]:
from fcdf_diagonal_frog.schemes.fcdf_dc import FCDF_DC_Solver

dc_solver = FCDF_DC_Solver()
out_dc = dc_solver.step(op, limiter, b, dt=gamma)
p_dc = out_dc["p_next"]
print(f"FCDF-DC: min(p)={p_dc.min():.3e}  mass={np.sum(p_dc)*grid.h:.6f}  "
      f"(predictor sweeps={out_dc['sweeps_predictor']}, corrector sweeps={out_dc['sweeps_corrector']})")


## Component 5: Active-set semismooth-Newton solver (Proposition 5)

Removes the Picard step-size restriction: tries the unlimited solve first, then updates
clamp patterns via banded solves until the pattern stabilizes.


In [ ]:
from fcdf_diagonal_frog.schemes.active_set import ActiveSetSolver

as_solver = ActiveSetSolver()
mu_bar = float(np.max(np.abs(mu)))
gamma_pic = FCDF_B_Solver.picard_contraction_bound(mu_bar, grid.h)
print(f"gamma_pic = {gamma_pic:.4e}")

for ratio in [0.5, 2, 10]:
    g = ratio * gamma_pic
    out_as = as_solver.solve(op, b, g)
    print(f"gamma/gamma_pic={ratio:>5}: pattern_updates={out_as['pattern_updates']}, "
          f"unlimited_accepted={out_as['unlimited_accepted']}, min(p)={out_as['p'].min():.2e}")


## Mini end-to-end reproduction: OU benchmark, small mesh (fast, no downloads)

This mirrors `train.py --debug`: n=51, T=0.02, FCDF-B on the Ornstein-Uhlenbeck benchmark.


In [ ]:
from fcdf_diagonal_frog.benchmarks.ou_process import OUBenchmark

bench = OUBenchmark(alpha=1.0, sigma=1.0, x0=0.5, v0=1e-2)
x_min, x_max = bench.domain(6)
grid_ou = Grid1D(x_min, x_max, 51)
mu_ou, D_ou = bench.drift(grid_ou.x), bench.diffusion(grid_ou.x)
op_ou = DFOperator.assemble(grid_ou, mu_ou, D_ou)
p0 = bench.initial_condition(grid_ou.x)

dt, T = 2e-4, 0.02
n_steps = int(round(T / dt))
p = p0.copy()
for step in range(n_steps):
    p = solver.step(op_ou, limiter, p, dt)["p"]
    if step % 20 == 0:
        print(f"  step {step:4d}/{n_steps}: min(p)={p.min():.2e}  mass={np.sum(p)*grid_ou.h:.6f}")

exact = bench.exact_density(grid_ou.x, T)
l1_err = grid_ou.h * np.sum(np.abs(p - exact))
print(f"\nFinal: min(p)={p.min():.3e}, L1 error vs. exact Gaussian = {l1_err:.4e}")

plt.figure(figsize=(7, 4))
plt.plot(grid_ou.x, p, label="FCDF-B (n=51)")
plt.plot(grid_ou.x, exact, "--", label="exact OU density")
plt.xlabel("x"); plt.ylabel("p(x, T)"); plt.legend(); plt.title("OU benchmark: FCDF-B vs. exact")
plt.tight_layout(); plt.show()


## Paper Results Comparison

Reported values from the SIR's `evaluation_protocol.reported_results`, alongside what this
reproduction actually measured when run at paper-scale mesh sizes via `evaluate.py`
(see `../comparison/comparison_report.md` for the full writeup):


In [ ]:
paper_vs_reproduction = {
    "Table 3 (OU spatial order, FCDF-B)": {
        "paper_reported": [1.78, 1.80, 1.59],
        "this_reproduction": [1.777, 1.803, 1.586],
        "match_quality": "near-exact",
    },
    "Table 7 (front, small dt, unlimited scheme min value)": {
        "paper_reported": -0.255,
        "this_reproduction": -0.123,
        "match_quality": "same sign/mechanism, different magnitude (different mesh/params)",
    },
    "Table 8 (active-set: gamma/gamma_pic=5, updates)": {
        "paper_reported": 0,
        "this_reproduction": 0,
        "match_quality": "exact",
    },
    "Table 2 (coverage condition (a), n=201)": {
        "paper_reported": True,
        "this_reproduction": True,
        "match_quality": "exact (qualitative)",
    },
}
for table, info in paper_vs_reproduction.items():
    print(f"\n{table}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

print("\nTo reproduce the full tables yourself, run:")
print("  python ../evaluate.py --config ../configs/config.yaml --table all")


## What to do next

1. **Full reproduction**: `python train.py --config configs/config.yaml` (paper-scale mesh)
2. **All tables**: `python evaluate.py --config configs/config.yaml --table all`
3. **Single-step demo**: `python inference.py --config configs/config.yaml --scheme fcdf_b`
4. **Tests**: `pytest ../tests/ -v` (14 tests covering conservation, positivity, Proposition 1/3/5)

**Top implementation caveats from the SIR / architecture plan (see `../README.md` for the full list):**
- `gamma_0`/`gamma_r` (linear positivity-window thresholds) are numerically bisected, not
  analytically derived — their closed forms live in an unavailable companion paper
  (SIR confidence 0.7).
- The sign-changing-drift stencil (needed for the OU benchmark) is our own resolution of an
  underspecified paper detail, validated indirectly via exact conservation and matched
  convergence orders (SIR confidence 0.6).
- The active-set solver's general mixed-pattern nonsingularity above `gamma_pic` is an
  **open question in the paper itself** — our implementation guards every solve and reports
  `converged=False` rather than assuming success (SIR confidence 0.82).
